In [1]:
import os

os.chdir("../..")

nmea_sentences_active_1 = {
    "GPVTG": "$GPVTG,,,,,,,,,N*30 $GPGGA,202233.00,,,,,0,00,99.99,,,,,,*64",
    "GPGSA": "$GPGSA,A,3,30,20,13,14,11,,,,,,,,4.17,3.14,2.74*01",
    "GPGSV_1": "$GPGSV,4,1,14,05,68,261,,07,37,058,,08,02,052,,09,02,096,*70",
    "GPGSV_2": "$GPGSV,4,2,14,11,04,201,,13,57,276,,14,28,138,29,15,21,283,*7E",
    "GPGSV_3": "$GPGSV,4,3,14,18,18,322,,20,53,190,28,21,56,171,,22,15,152,*7B",
    "GPGSV_4": "$GPGSV,4,4,14,27,04,019,,30,69,085,*74",
    "GPGLL": "$GPGLL,,,,,202233.00,V,N*48",
    "GPRMC": "$GPRMC,200517.00,A,5211.77084,N,00213.26972,W,0.117,,030625,,,A*65",
}

nmea_sentences_active_2 = {
    "GPVTG": "$GPVTG,,,,,,,,,N*30 $GPGGA,202233.00,,,,,0,00,99.99,,,,,,*64",
    "GPGSA": "$GPGSA,A,3,30,20,13,14,11,,,,,,,,4.17,3.14,2.74*01",
    "GPGSV_1": "$GPGSV,4,1,14,05,68,261,,07,37,058,,08,02,052,,09,02,096,*70",
    "GPGSV_2": "$GPGSV,4,2,14,11,04,201,,13,57,276,,14,28,138,29,15,21,283,*7E",
    "GPGSV_3": "$GPGSV,4,3,14,18,18,322,,20,53,190,28,21,56,171,,22,15,152,*7B",
    "GPGSV_4": "$GPGSV,4,4,14,27,04,019,,30,69,085,*74",
    "GPGLL": "$GPGLL,,,,,202233.00,V,N*48",
    "GPRMC": "$GPRMC,200518.00,A,5212.77195,N,00212.27083,W,4.000,,030625,,,A*6C",
}

nmea_sentences_void = {
    "GPVTG": "$GPVTG,,,,,,,,,N*30 $GPGGA,202233.00,,,,,0,00,99.99,,,,,,*64",
    "GPGSA": "$GPGSA,A,1,,,,,,,,,,,,,99.99,99.99,99.99*30 ",
    "GPGSV_1": "$GPGSV,4,1,14,05,68,261,,07,37,058,,08,02,052,,09,02,096,*70",
    "GPGSV_2": "$GPGSV,4,2,14,11,04,201,,13,57,276,,14,28,138,29,15,21,283,*7E",
    "GPGSV_3": "$GPGSV,4,3,14,18,18,322,,20,53,190,28,21,56,171,,22,15,152,*7B",
    "GPGSV_4": "$GPGSV,4,4,14,27,04,019,,30,69,085,*74",
    "GPGLL": "$GPGLL,,,,,202233.00,V,N*48",
    "GPRMC": "$GPRMC,200520.00,V,,,,,,,090625,,,N*70",
}

nmea_sentences_active_3 = {
    "GPVTG": "$GPVTG,,,,,,,,,N*30 $GPGGA,202233.00,,,,,0,00,99.99,,,,,,*64",
    "GPGSA": "$GPGSA,A,3,30,20,13,14,11,,,,,,,,4.17,3.14,2.74*01",
    "GPGSV_1": "$GPGSV,4,1,14,05,68,261,,07,37,058,,08,02,052,,09,02,096,*70",
    "GPGSV_2": "$GPGSV,4,2,14,11,04,201,,13,57,276,,14,28,138,29,15,21,283,*7E",
    "GPGSV_3": "$GPGSV,4,3,14,18,18,322,,20,53,190,28,21,56,171,,22,15,152,*7B",
    "GPGSV_4": "$GPGSV,4,4,14,27,04,019,,30,69,085,*74",
    "GPGLL": "$GPGLL,,,,,202233.00,V,N*48",
    "GPRMC": "$GPRMC,200519.00,A,5213.77306,N,00211.27194,W,4.000,,030625,,,A*60",
}

from src.kelder_api.components.gps_new.interface import GPSInterface
from src.kelder_api.configuration.settings import get_settings
from src.kelder_api.components.redis_client.redis_client import RedisClient
from datetime import datetime, timezone

redis_client = RedisClient()
gps_interface = GPSInterface(redis_client)

Add in course over ground code:

In [14]:
velocity_calculator = VelocityCalculator(
    gps_interface=gps_interface, redis_client=redis_client
)

Testing

In [32]:
from src.kelder_api.components.velocity.service import VelocityCalculator

# Clear the redis GPS set
async with redis_client.get_connection() as redis:
    await redis.delete("sensor:ts:GPS")
    await redis.delete("sensor:ts:VELOCITY")
    size = await redis.zcard("sensor:ts:GPS")
    print(f"Redis set size: {size}")
    await redis.zremrangebyrank("sensor:ts:GPS", 0, -1)

# gps_latest = await gps_interface.read_gps_all_history()
# for gps_measurement in gps_latest:
#     print(gps_measurement)

await gps_interface.stream_serial_data(list(nmea_sentences_active_1.values()))
await gps_interface.stream_serial_data(list(nmea_sentences_active_2.values()))
await gps_interface.stream_serial_data(list(nmea_sentences_active_3.values()))
#await gps_interface.stream_serial_data(list(nmea_sentences_void.values()))

now = datetime(2025, 6, 3, 20, 5, 22, tzinfo=timezone.utc)


await velocity_calculator.calculate_gps_velocity(now)

Redis set size: 0
[GPSRedisData(timestamp=datetime.datetime(2025, 6, 3, 20, 5, 19, tzinfo=TzInfo(UTC)), status=<GPSStatus.ACTIVE: 'A'>, latitude_nmea='5213.77306', longitude_nmea='00211.27194', active_prn=[30, 20, 13, 14, 11], hdop=3.14, satellites_in_view={5: SatelliteInfomation(prn=5, elevation=68, azimuth=261, snr=None), 7: SatelliteInfomation(prn=7, elevation=37, azimuth=58, snr=None), 8: SatelliteInfomation(prn=8, elevation=2, azimuth=52, snr=None), 9: SatelliteInfomation(prn=9, elevation=2, azimuth=96, snr=None), 11: SatelliteInfomation(prn=11, elevation=4, azimuth=201, snr=None), 13: SatelliteInfomation(prn=13, elevation=57, azimuth=276, snr=None), 14: SatelliteInfomation(prn=14, elevation=28, azimuth=138, snr=29.0), 15: SatelliteInfomation(prn=15, elevation=21, azimuth=283, snr=None), 18: SatelliteInfomation(prn=18, elevation=18, azimuth=322, snr=None), 20: SatelliteInfomation(prn=20, elevation=53, azimuth=190, snr=28.0), 21: SatelliteInfomation(prn=21, elevation=56, azimuth=17

In [15]:
await velocity_calculator.read_velocity_latest(active=True)

GPSVelocity(timestamp=datetime.datetime(2025, 6, 3, 20, 5, 20, tzinfo=TzInfo(UTC)), speed_over_ground=2.1745691217861074, course_over_ground=148.5530217599021, number_of_measurements=3)

In [29]:
await velocity_calculator.read_velocity_all(active=True)

[GPSVelocity(timestamp=datetime.datetime(2025, 6, 3, 20, 5, 23, tzinfo=TzInfo(UTC)), speed_over_ground=2.1745691217861074, course_over_ground=148.5530217599021, number_of_measurements=3),
 GPSVelocity(timestamp=datetime.datetime(2025, 6, 3, 20, 5, 22, tzinfo=TzInfo(UTC)), speed_over_ground=2.1746802346323433, course_over_ground=148.54823389820604, number_of_measurements=2)]